# Tarea 6 — Big Data y Búsqueda Semántica

**Asignatura:** Gestión de Datos  
**Institución:** Pontificia Universidad Javeriana  
**Departamento:** Departamento de Ingeniería de Sistemas  

---

### 👥 Identificación del Grupo y del Dataset

- **Integrantes:** [Nombres completos de los integrantes]
- **Descripción del Dataset (PDF externo elegido):** [Breve descripción de 2 a 3 líneas sobre el documento PDF analizado: tema, procedencia y relevancia]

---

En este taller indexaremos un documento PDF externo de su elección: extraeremos su texto, lo dividiremos en fragmentos (*chunks*), generaremos representaciones vectoriales densas (*embeddings*) comparando **dos modelos locales de Hugging Face** (uno sencillo y uno intermedio) y realizaremos búsquedas semánticas sobre una base de datos vectorial en memoria (**ChromaDB**).

## 1. Preparar el entorno (3 minutos)

Instalamos las librerías necesarias y creamos automáticamente la carpeta `input` en el panel de archivos de la izquierda (donde subirá su PDF). Para Hugging Face y ChromaDB no se requiere ninguna clave de API ni registro previo.

In [ ]:
!pip -q install -U pypdf sentence-transformers chromadb

In [ ]:
from pathlib import Path
from time import perf_counter
import re
import numpy as np
import pandas as pd
from IPython.display import display

CARPETA_INPUT = Path('/content/input')
CARPETA_INPUT.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 3

## 2. Cargar el documento PDF externo

Suba el archivo PDF de su elección ejecutando la siguiente celda (o cárguelo directamente en la carpeta `input` recién creada en el panel de archivos a la izquierda).

In [ ]:
from google.colab import files

rutas_pdf = sorted(CARPETA_INPUT.glob('*.pdf'))
if not rutas_pdf:
    print('Por favor seleccione y suba su archivo PDF:')
    subidos = files.upload()
    for nombre, contenido in subidos.items():
        if nombre.lower().endswith('.pdf'):
            (CARPETA_INPUT / Path(nombre).name).write_bytes(contenido)
            print(f'PDF guardado en la carpeta input: {nombre}')
        else:
            print(f'Omitido (no es un archivo PDF): {nombre}')
    rutas_pdf = sorted(CARPETA_INPUT.glob('*.pdf'))

assert rutas_pdf, 'No se encontró ningún archivo PDF en la carpeta input. Por favor suba un archivo.'
print('Archivos PDF listos para procesar:', [p.name for p in rutas_pdf])

## 3. Extraer páginas y crear chunks (7 minutos)

Convertimos el PDF a texto preservando número de página y creamos fragmentos con solapamiento (*overlap*) para no cortar oraciones a la mitad.

In [ ]:
from pypdf import PdfReader

def extraer_paginas(rutas):
    paginas = []
    for ruta in rutas:
        lector = PdfReader(str(ruta))
        for numero, pagina in enumerate(lector.pages, start=1):
            texto = re.sub(r'\s+', ' ', pagina.extract_text() or '').strip()
            paginas.append({
                'documento': ruta.name,
                'pagina': numero,
                'texto': texto,
                'caracteres': len(texto),
            })
    return paginas

paginas = extraer_paginas(rutas_pdf)
df_paginas = pd.DataFrame(paginas)
display(df_paginas[['documento', 'pagina', 'caracteres']].head(10))
assert df_paginas.caracteres.sum() > 0, 'El PDF no contiene texto extraíble (puede requerir OCR).'

In [ ]:
def crear_chunks(paginas, chunk_size=800, overlap=120):
    assert 0 <= overlap < chunk_size
    chunks = []
    for pagina in paginas:
        texto, inicio, numero_chunk = pagina['texto'], 0, 1
        while inicio < len(texto):
            fin = min(inicio + chunk_size, len(texto))
            if fin < len(texto):
                corte = texto.rfind(' ', inicio + chunk_size // 2, fin)
                if corte > inicio:
                    fin = corte
            fragmento = texto[inicio:fin].strip()
            if fragmento:
                base = re.sub(r'[^a-zA-Z0-9]+', '-', Path(pagina['documento']).stem).strip('-').lower()
                chunks.append({
                    'chunk_id': f'{base}-p{pagina["pagina"]:03d}-c{numero_chunk:03d}',
                    'documento': pagina['documento'],
                    'pagina': int(pagina['pagina']),
                    'texto': fragmento,
                })
                numero_chunk += 1
            if fin >= len(texto):
                break
            inicio = max(fin - overlap, inicio + 1)
    return chunks

chunks = crear_chunks(paginas, CHUNK_SIZE, CHUNK_OVERLAP)
df_chunks = pd.DataFrame(chunks)
display(df_chunks[['chunk_id', 'documento', 'pagina', 'texto']].head())
print(f'{len(paginas)} páginas → {len(chunks)} chunks generados')
assert df_chunks.chunk_id.is_unique

## 4. Generar embeddings (10 minutos)

Utilizaremos y compararemos dos modelos locales de Hugging Face optimizados para español. Al correr localmente, no requieren claves de API ni envían sus datos a terceros.

1. **Modelo Sencillo:** `hiiamsid/sentence_similarity_spanish_es` (rápido y ligero).
2. **Modelo Intermedio:** `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (robusto y multilingüe).

In [ ]:
from sentence_transformers import SentenceTransformer
from time import perf_counter

print('--- Modelo Sencillo ---')
inicio_s = perf_counter()
modelo_sencillo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')
emb_sencillo = modelo_sencillo.encode(
    [c['texto'] for c in chunks],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
).astype('float32')
tiempo_sencillo = perf_counter() - inicio_s
print('Sencillo listo:', emb_sencillo.shape, f'{tiempo_sencillo:.2f} s', f'{emb_sencillo.nbytes / 1e6:.3f} MB')

In [ ]:
print('\n--- Modelo Intermedio ---')
inicio_i = perf_counter()
modelo_intermedio = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
emb_intermedio = modelo_intermedio.encode(
    [c['texto'] for c in chunks],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
).astype('float32')
tiempo_intermedio = perf_counter() - inicio_i
print('Intermedio listo:', emb_intermedio.shape, f'{tiempo_intermedio:.2f} s', f'{emb_intermedio.nbytes / 1e6:.3f} MB')

## 5. Indexar en ChromaDB y buscar por similitud semántica (8 minutos)

In [ ]:
import chromadb

cliente_chroma = chromadb.Client()
ids = [c['chunk_id'] for c in chunks]
documentos = [c['texto'] for c in chunks]
metadatos = [{'documento': c['documento'], 'pagina': c['pagina']} for c in chunks]

def crear_coleccion(nombre, embeddings):
    try:
        cliente_chroma.delete_collection(nombre)
    except Exception:
        pass
    coleccion = cliente_chroma.create_collection(nombre, metadata={'hnsw:space': 'cosine'})
    coleccion.add(ids=ids, documents=documentos, metadatas=metadatos, embeddings=embeddings.tolist())
    return coleccion

coleccion_sencillo = crear_coleccion('chunks_sencillo', emb_sencillo)
coleccion_intermedio = crear_coleccion('chunks_intermedio', emb_intermedio)
print('Documentos indexados (Sencillo):', coleccion_sencillo.count())
print('Documentos indexados (Intermedio):', coleccion_intermedio.count())

In [ ]:
# Defina o adapte entre 2 y 4 consultas de interés para el contenido de su propio PDF
CONSULTAS = [
    '¿Cuáles son los temas o conceptos principales tratados en el documento?',
    '¿Qué conclusiones, resultados o metodologías clave se destacan?',
    '¿Qué problemáticas, antecedentes o retos se exponen?',
]

def vector_consulta(texto, modelo):
    return modelo.encode([texto], normalize_embeddings=True)[0].astype('float32')

def buscar(coleccion, vector, consulta, nombre_modelo):
    salida = coleccion.query(
        query_embeddings=[vector.tolist()], n_results=TOP_K,
        include=['documents', 'metadatas', 'distances'],
    )
    return pd.DataFrame([{
        'modelo': nombre_modelo, 'consulta': consulta, 'rank': i + 1,
        'chunk_id': salida['ids'][0][i],
        'pagina': salida['metadatas'][0][i]['pagina'],
        'similitud': round(1 - salida['distances'][0][i], 4),
        'texto': salida['documents'][0][i],
    } for i in range(len(salida['ids'][0]))])

In [ ]:
resultados = []
for consulta in CONSULTAS:
    texto_q = consulta if isinstance(consulta, str) else consulta['consulta']
    
    vec_s = vector_consulta(texto_q, modelo_sencillo)
    resultados.append(buscar(coleccion_sencillo, vec_s, texto_q, 'Sencillo'))
    
    vec_i = vector_consulta(texto_q, modelo_intermedio)
    resultados.append(buscar(coleccion_intermedio, vec_i, texto_q, 'Intermedio'))

df_resultados = pd.concat(resultados, ignore_index=True)
display(df_resultados[['modelo', 'consulta', 'rank', 'pagina', 'similitud', 'texto']])

## 6. Métricas de almacenamiento y rendimiento (6 minutos)

In [ ]:
filas_metricas = [
    {
        'modelo': 'Sencillo', 'dimensiones': emb_sencillo.shape[1],
        'tiempo_indexacion_s': round(tiempo_sencillo, 2), 'memoria_vectores_mb': round(emb_sencillo.nbytes / 1e6, 3),
        'similitud_promedio_top1': round(df_resultados[(df_resultados['rank'] == 1) & (df_resultados['modelo'] == 'Sencillo')]['similitud'].mean(), 4),
    },
    {
        'modelo': 'Intermedio', 'dimensiones': emb_intermedio.shape[1],
        'tiempo_indexacion_s': round(tiempo_intermedio, 2), 'memoria_vectores_mb': round(emb_intermedio.nbytes / 1e6, 3),
        'similitud_promedio_top1': round(df_resultados[(df_resultados['rank'] == 1) & (df_resultados['modelo'] == 'Intermedio')]['similitud'].mean(), 4),
    }
]
df_metricas = pd.DataFrame(filas_metricas)
display(df_metricas)

top_sencillo = df_resultados[df_resultados.modelo == 'Sencillo'].groupby('consulta').chunk_id.apply(set)
top_intermedio = df_resultados[df_resultados.modelo == 'Intermedio'].groupby('consulta').chunk_id.apply(set)
solapamiento = pd.DataFrame({
    'consulta': top_sencillo.index,
    'coincidencias_top3': [len(top_sencillo[q] & top_intermedio[q]) for q in top_sencillo.index],
})
display(solapamiento)

## 7. Reto Avanzado: Modelos más pesados y granularidad (Opcional)

Si desea profundizar, intente utilizar el modelo `sentence-transformers/LaBSE` (diseñado por Google, 471M de parámetros) y evalúe cómo cambian los resultados. Además, a continuación evaluamos el impacto de reducir el tamaño de chunk a 400 caracteres con solapamiento de 60 sobre la primera consulta.

In [ ]:
chunks_cortos = crear_chunks(paginas, chunk_size=400, overlap=60)
emb_sencillo_cortos = modelo_sencillo.encode(
    [c['texto'] for c in chunks_cortos],
    normalize_embeddings=True, batch_size=32, show_progress_bar=False,
).astype('float32')

ids_originales, documentos_originales, metadatos_originales = ids, documentos, metadatos
ids = [c['chunk_id'] for c in chunks_cortos]
documentos = [c['texto'] for c in chunks_cortos]
metadatos = [{'documento': c['documento'], 'pagina': c['pagina']} for c in chunks_cortos]
coleccion_corta = crear_coleccion('chunks_corta', emb_sencillo_cortos)

consulta_reto = CONSULTAS[0] if isinstance(CONSULTAS[0], str) else CONSULTAS[0]['consulta']
display(buscar(coleccion_corta, vector_consulta(consulta_reto, modelo_sencillo), consulta_reto, 'Sencillo chunks=400'))
ids, documentos, metadatos = ids_originales, documentos_originales, metadatos_originales
print(f'Chunks totales: {len(chunks)} (800 car.) → {len(chunks_cortos)} (400 car.)')

## 8. Evidencia Ejercicio 2 — Google Cloud BigQuery

Pegue en esta sección la captura de pantalla de su consulta ejecutada en BigQuery Studio y responda a la pregunta:

1. **Captura de Pantalla:**  
   *(Pegue aquí la imagen con la consulta y la tabla de resultados de BigQuery)*

2. **Respuesta:**  
   - **¿Cuáles son los 5 nombres de niños más comunes en 2014 según el dataset?**:  
   - **¿Cuántos registros tuvo el nombre número 1?**: